In [6]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
import logging
from datetime import datetime, timedelta
import numpy as np

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_database_connection():
    """Create and return a database connection."""
    return mysql.connector.connect(
        host="10.200.200.107",
        user="scriptuser",
        password="!!Driveline11",
        database="theia_pitching_db"
    )

def check_data_completeness(df, table_name, key_columns):
    """
    Check for missing/incomplete data in a given DataFrame.
    Logs the sum of nulls per column, unique non-null counts, and data types.
    If a 'time' column is present, calculates basic time gap statistics.
    """
    logger.info(f"\nChecking data completeness for {table_name}:")
    
    # Count null values per column
    null_counts = df.isnull().sum()
    if null_counts.any():
        logger.warning(f"Null values found in {table_name}:")
        logger.warning(null_counts[null_counts > 0])
    else:
        logger.info("No null values found.")
    
    # Report unique non-null counts and column types
    for col in df.columns:
        unique_count = df[col].nunique(dropna=True)
        col_type = df[col].dtype
        logger.info(f"Column '{col}': Type = {col_type}, Unique non-null values = {unique_count}")
    
    # If a 'time' column exists, provide basic statistics on time gaps grouped by session_trial.
    if 'time' in df.columns:
        time_stats = df.groupby('session_trial')['time'].apply(lambda x: x.diff().describe())
        logger.info(f"Time series statistics for {table_name} (averaged across sessions):\n{time_stats.mean()}")

def validate_join(df_before, df_after, join_type, key_columns):
    """
    Validate that an inner join did not lose rows or key information.
    Logs row counts, unique session_trial counts, and warns if any key columns' unique counts differ.
    """
    logger.info(f"\nValidating {join_type} join:")
    
    before_counts = {
        'rows': len(df_before),
        'session_trials': df_before['session_trial'].nunique(),
        'times': df_before['time'].nunique() if 'time' in df_before.columns else None
    }
    after_counts = {
        'rows': len(df_after),
        'session_trials': df_after['session_trial'].nunique(),
        'times': df_after['time'].nunique() if 'time' in df_after.columns else None
    }
    
    logger.info(f"Rows before: {before_counts['rows']}, after: {after_counts['rows']}")
    logger.info(f"Unique trials before: {before_counts['session_trials']}, after: {after_counts['session_trials']}")
    
    # Check if any trials were lost in the join
    if 'session_trial' in df_before.columns:
        lost_trials = set(df_before['session_trial'].unique()) - set(df_after['session_trial'].unique())
        if lost_trials:
            logger.warning(f"Lost trials in {join_type} join: {lost_trials}")
    
    # Compare unique counts for each key column
    for col in key_columns:
        if col in df_before.columns and col in df_after.columns:
            before_unique = df_before[col].nunique()
            after_unique = df_after[col].nunique()
            if before_unique != after_unique:
                logger.warning(f"Unique count for '{col}' changed: {before_unique} -> {after_unique}")
    
    return before_counts['rows'] == after_counts['rows']

def integrate_joint_data():
    """
    Loads and validates joint angles, forces, moments, and velocities for sessions
    from the last month. Joins the datasets stepwise with validations and then
    merges in trial and session metadata.
    """
    conn = get_database_connection()
    
    # Define a time filter for sessions (last month)
    one_month_ago = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
    
    logger.info("Loading sessions from the last month...")
    sessions = pd.read_sql(f"""
        SELECT 
            session,
            date,
            level,
            lab,
            height_meters,
            mass_kilograms
        FROM `sessions`
        WHERE date >= '{one_month_ago}'
    """, conn)
    
    logger.info("Loading trials for these sessions...")
    sessions_list = sessions['session'].tolist()
    trials = pd.read_sql(f"""
        SELECT 
            session_trial,
            session,
            trial,
            pitch_type,
            handedness,
            filename
        FROM `trials`
        WHERE session IN ({','.join(map(str, sessions_list))})
    """, conn)
    
    valid_session_trials = tuple(trials['session_trial'].unique())
    
    logger.info("Loading joint data tables...")
    joint_angles = pd.read_sql(f"""
        SELECT * FROM `joint_angles`
        WHERE session_trial IN {valid_session_trials}
    """, conn)
    
    joint_forces = pd.read_sql(f"""
        SELECT * FROM `joint_forces`
        WHERE session_trial IN {valid_session_trials}
    """, conn)
    
    joint_moments = pd.read_sql(f"""
        SELECT * FROM `joint_moments`
        WHERE session_trial IN {valid_session_trials}
    """, conn)
    
    joint_velos = pd.read_sql(f"""
        SELECT * FROM `joint_velos`
        WHERE session_trial IN {valid_session_trials}
    """, conn)
    
    # Validate completeness of each base table
    key_columns = ['session_trial', 'time']
    check_data_completeness(joint_angles, 'joint_angles', key_columns)
    check_data_completeness(joint_forces, 'joint_forces', key_columns)
    check_data_completeness(joint_moments, 'joint_moments', key_columns)
    check_data_completeness(joint_velos, 'joint_velos', key_columns)
    
    # Perform inner joins with validation at each step.
    logger.info("Joining joint_angles and joint_forces...")
    angles_forces = pd.merge(
        joint_angles,
        joint_forces,
        on=['session_trial', 'time'],
        how='inner'
    )
    validate_join(joint_angles, angles_forces, "angles-forces", key_columns)
    
    logger.info("Adding joint_moments to the join...")
    angles_forces_moments = pd.merge(
        angles_forces,
        joint_moments,
        on=['session_trial', 'time'],
        how='inner'
    )
    validate_join(angles_forces, angles_forces_moments, "moments", key_columns)
    
    logger.info("Adding joint_velos to the join...")
    all_joint_data = pd.merge(
        angles_forces_moments,
        joint_velos,
        on=['session_trial', 'time'],
        how='inner'
    )
    validate_join(angles_forces_moments, all_joint_data, "velocities", key_columns)
    
    logger.info("Merging with trials metadata...")
    joint_with_trials = pd.merge(
        all_joint_data,
        trials,
        on='session_trial',
        how='left'
    )
    validate_join(all_joint_data, joint_with_trials, "trials", key_columns + ['session', 'trial', 'pitch_type'])
    
    logger.info("Merging with sessions metadata...")
    final_dataset = pd.merge(
        joint_with_trials,
        sessions,
        on='session',
        how='left'
    )
    validate_join(joint_with_trials, final_dataset, "sessions", key_columns + ['session', 'date', 'level'])
    
    logger.info("\nFinal Integrated Dataset Summary:")
    logger.info(f"Total rows: {len(final_dataset)}")
    logger.info(f"Unique trials: {final_dataset['session_trial'].nunique()}")
    logger.info(f"Unique sessions: {final_dataset['session'].nunique()}")
    logger.info(f"Date range: {final_dataset['date'].min()} to {final_dataset['date'].max()}")
    
    # Check for missing data in critical columns
    critical_columns = ['session_trial', 'time', 'session', 'date', 'pitch_type']
    missing_data = final_dataset[critical_columns].isnull().sum()
    if missing_data.any():
        logger.warning("Missing data in critical columns:")
        logger.warning(missing_data[missing_data > 0])
    
    return final_dataset

def run_final_query():
    """
    Executes the final comprehensive query which uses several CTEs to combine biomechanical
    data (with phase segmentation, energy, loads, kinematics, and composite injury scores)
    into a single result set.
    
    The query now begins with a filtering CTE (date_filtered_sessions) that filters sessions
    based on date. (Several date filters are provided as comments; by default, no filter is applied.)
    """
    conn = get_database_connection()
    
    # Updated final_query with filtering CTE integrated.
    final_query = r"""
    /*
    ================================================================================
    -- NOTES / GOALS FOR THIS QUERY:
    --
    -- GOALS:
    -- 1. Create a comprehensive biomechanical profile for each pitch.
    -- 2. Segment the pitch into phases (e.g., Wind-Up, Stride, Arm Cocking, etc.)
    -- 3. Pull kinetic, kinematic, and energy data to assess potential injury risk.
    -- 4. Compute additional metrics (e.g., cumulative load, variability, trunk–pelvis
    --    dissociation) for deeper analysis.
    -- 5. Join composite injury risk scores (e.g., pitch speed, shoulder rotation velocity)
    --    for cross-validation.
    --
    -- TABLES:
    -- • events, energetics, force_plates, joint_forces, joint_moments,
    --   joint_angles, joint_velos, poi.
    --
    -- METRICS:
    -- - Phase durations, energy metrics (upper & lower limb), joint kinematics,
    --   loads, ground reaction forces, computed metrics (trunk–pelvis dissociation,
    --   cumulative lead force, variability), and composite injury risk scores.
    ================================================================================
    */
    
    -- New Filtering CTE (date_filtered_sessions)
    WITH date_filtered_sessions AS (
        SELECT
            t.session_trial,
            s.date,
            s.session
        FROM `trials` t
        JOIN `sessions` s ON t.session = s.session
        WHERE
            -- Choose ONE of these date filters by uncommenting:
            
            -- 1. For a specific date (replace YYYY-MM-DD with your date)
            -- s.date = '2024-02-17'
            
            -- 2. For a specific month (replace YYYY-MM-DD with your year and month)
            -- s.date BETWEEN '2024-02-01' AND LAST_DAY('2024-02-01')
            
            -- 3. For the last day
            -- s.date = DATE_SUB(CURDATE(), INTERVAL 1 DAY)
            
            -- 4. For the last 30 days
            -- s.date >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
            
            -- 5. For the last calendar month
            -- s.date BETWEEN DATE_FORMAT(DATE_SUB(CURDATE(), INTERVAL 1 MONTH), '%Y-%m-01') 
            --    AND LAST_DAY(DATE_SUB(CURDATE(), INTERVAL 1 MONTH))
            
            1=1  -- Default condition if no filter is selected
    ),
    
    -- 1. Pitch Phases CTE: Now joined to the filtered sessions.
    pitch_phases AS (
        SELECT 
            e.session_trial,
            e.PKH_time AS phase_pkh,
            e.FP_v5_time AS phase_fp,
            e.MER_time AS phase_mer,
            e.BR_time AS phase_br,
            e.MAD_time AS phase_mad,
            (e.FP_v5_time - e.PKH_time) AS duration_pkh_to_fp,
            (e.MER_time - e.FP_v5_time) AS duration_fp_to_mer,
            (e.BR_time - e.MER_time) AS duration_mer_to_br,
            (e.MAD_time - e.BR_time) AS duration_br_to_mad
        FROM `events` e
        INNER JOIN date_filtered_sessions dfs ON e.session_trial = dfs.session_trial
    ),
    
    -- 2. Energy Metrics CTE with additional lower limb columns
    energy_metrics AS (
        SELECT 
            session_trial,
            time,
            shoulder_energy_transfer,
            shoulder_energy_generation,
            elbow_energy_transfer,
            elbow_energy_generation,
            lead_ankle_energy_transfer,
            lead_ankle_energy_generation,
            rear_ankle_energy_transfer,
            rear_ankle_energy_generation,
            lead_knee_energy_transfer,
            lead_knee_energy_generation,
            rear_knee_energy_transfer,
            rear_knee_energy_generation,
            lead_hip_energy_transfer,
            lead_hip_energy_generation,
            rear_hip_energy_transfer,
            rear_hip_energy_generation
        FROM `energetics`
    ),
    
    -- 3. Force Data CTE
    force_data AS (
        SELECT
            session_trial,
            time,
            lead_force_x,
            lead_force_y,
            lead_force_z,
            lead_force_mag,
            rear_force_x,
            rear_force_y,
            rear_force_z,
            rear_force_mag
        FROM `force_plates`
    ),
    
    -- 4. Joint Loads CTE
    joint_loads AS (
        SELECT
            jf.session_trial,
            jf.time,
            jf.elbow_force_x,
            jf.elbow_force_y,
            jf.elbow_force_z,
            jf.shoulder_upper_arm_force_x,
            jf.shoulder_upper_arm_force_y,
            jf.shoulder_upper_arm_force_z,
            jm.elbow_moment_x,
            jm.elbow_moment_y,
            jm.elbow_moment_z,
            jm.shoulder_thorax_moment_x,
            jm.shoulder_thorax_moment_y,
            jm.shoulder_thorax_moment_z
        FROM `joint_forces` jf
        JOIN `joint_moments` jm 
          ON jf.session_trial = jm.session_trial 
         AND jf.time = jm.time
    ),
    
    -- 5. Biomechanics with Phase CTE (with trunk–pelvis dissociation)
    biomech_with_phase AS (
        SELECT 
            ja.session_trial,
            ja.time,
            ja.shoulder_angle_x,
            ja.shoulder_angle_y,
            ja.shoulder_angle_z,
            ja.elbow_angle_x,
            ja.elbow_angle_y,
            ja.elbow_angle_z,
            jv.shoulder_velo_x,
            jv.shoulder_velo_y,
            jv.shoulder_velo_z,
            jv.elbow_velo_x,
            jv.elbow_velo_y,
            jv.elbow_velo_z,
            ja.torso_angle_x,
            ja.torso_angle_y,
            ja.torso_angle_z,
            ja.pelvis_angle_x,
            ja.pelvis_angle_y,
            ja.pelvis_angle_z,
            jv.torso_velo_x,
            jv.torso_velo_y,
            jv.torso_velo_z,
            em.shoulder_energy_transfer,
            em.shoulder_energy_generation,
            em.elbow_energy_transfer,
            em.elbow_energy_generation,
            em.lead_ankle_energy_transfer,
            em.lead_ankle_energy_generation,
            em.rear_ankle_energy_transfer,
            em.rear_ankle_energy_generation,
            em.lead_knee_energy_transfer,
            em.lead_knee_energy_generation,
            em.rear_knee_energy_transfer,
            em.rear_knee_energy_generation,
            em.lead_hip_energy_transfer,
            em.lead_hip_energy_generation,
            em.rear_hip_energy_transfer,
            em.rear_hip_energy_generation,
            jl.elbow_force_x,
            jl.elbow_force_y,
            jl.elbow_force_z,
            jl.shoulder_upper_arm_force_x,
            jl.shoulder_upper_arm_force_y,
            jl.shoulder_upper_arm_force_z,
            jl.elbow_moment_x,
            jl.elbow_moment_y,
            jl.elbow_moment_z,
            jl.shoulder_thorax_moment_x,
            jl.shoulder_thorax_moment_y,
            jl.shoulder_thorax_moment_z,
            fd.lead_force_mag,
            fd.rear_force_mag,
            ABS(ja.torso_angle_z - ja.pelvis_angle_z) AS trunk_pelvis_dissociation,
            CASE 
                WHEN ja.time <= pp.phase_pkh THEN 'Wind-Up'
                WHEN ja.time <= pp.phase_fp THEN 'Stride'
                WHEN ja.time <= pp.phase_mer THEN 'Arm Cocking'
                WHEN ja.time <= pp.phase_br THEN 'Arm Acceleration'
                WHEN ja.time <= pp.phase_mad THEN 'Arm Deceleration'
                ELSE 'Follow Through'
            END AS pitch_phase,
            pp.duration_pkh_to_fp,
            pp.duration_fp_to_mer,
            pp.duration_mer_to_br,
            pp.duration_br_to_mad
        FROM `joint_angles` ja
        JOIN pitch_phases pp 
          ON ja.session_trial = pp.session_trial
        JOIN `joint_velos` jv 
          ON ja.session_trial = jv.session_trial 
         AND ja.time = jv.time
        LEFT JOIN energy_metrics em 
          ON ja.session_trial = em.session_trial 
         AND ja.time = em.time
        LEFT JOIN joint_loads jl 
          ON ja.session_trial = jl.session_trial 
         AND ja.time = jl.time
        LEFT JOIN force_data fd 
          ON ja.session_trial = fd.session_trial 
         AND ja.time = fd.time
    ),
    
    -- 6. Final Metrics CTE: Aggregates biomechanical data and joins composite scores from poi.
    final_metrics AS (
        SELECT 
            bp.session_trial,
            bp.pitch_phase,
            COUNT(*) AS frames_in_phase,
            AVG(shoulder_angle_x) AS avg_shoulder_flex_ext,
            AVG(shoulder_angle_y) AS avg_shoulder_abd_add,
            AVG(shoulder_angle_z) AS avg_shoulder_rotation,
            AVG(elbow_angle_x) AS avg_elbow_flex_ext,
            MAX(ABS(shoulder_velo_z)) AS max_shoulder_rotation_speed,
            MAX(ABS(elbow_velo_x)) AS max_elbow_extension_speed,
            AVG(torso_angle_x) AS avg_torso_flex,
            AVG(torso_angle_y) AS avg_torso_lateral_tilt,
            MAX(ABS(torso_velo_z)) AS max_torso_rotation_speed,
            AVG(shoulder_energy_transfer) AS avg_shoulder_energy_transfer,
            MAX(shoulder_energy_generation) AS max_shoulder_energy_generation,
            AVG(elbow_energy_transfer) AS avg_elbow_energy_transfer,
            MAX(elbow_energy_generation) AS max_elbow_energy_generation,
            MAX(ABS(elbow_moment_y)) AS max_elbow_varus_moment,
            MAX(ABS(shoulder_thorax_moment_x)) AS max_shoulder_distraction_force,
            MAX(lead_force_mag) AS max_lead_leg_force,
            MAX(rear_force_mag) AS max_push_off_force,
            SUM(lead_force_mag) AS cumulative_lead_force,
            STDDEV(shoulder_angle_z) AS shoulder_rotation_variability,
            AVG(trunk_pelvis_dissociation) AS avg_trunk_pelvis_dissociation,
            AVG(duration_pkh_to_fp) AS avg_duration_pkh_to_fp,
            AVG(duration_fp_to_mer) AS avg_duration_fp_to_mer,
            AVG(duration_mer_to_br) AS avg_duration_mer_to_br,
            AVG(duration_br_to_mad) AS avg_duration_br_to_mad,
            p.pitch_speed_mph,
            p.max_shoulder_internal_rotational_velo
        FROM biomech_with_phase bp
        LEFT JOIN poi p 
          ON bp.session_trial = p.session_trial
        GROUP BY bp.session_trial, bp.pitch_phase
    )
    
    SELECT * FROM final_metrics
    ORDER BY session_trial, 
        CASE pitch_phase
            WHEN 'Wind-Up' THEN 1
            WHEN 'Stride' THEN 2
            WHEN 'Arm Cocking' THEN 3
            WHEN 'Arm Acceleration' THEN 4
            WHEN 'Arm Deceleration' THEN 5
            WHEN 'Follow Through' THEN 6
        END;
    """
    logger.info("Executing final SQL query...")
    final_result = pd.read_sql(final_query, conn)
    conn.close()
    return final_result

def check_summary_dataset(df, dataset_name="Final Summarized Dataset"):
    """
    Perform quality checks on the summarized dataset.
    Logs the sum and percentage of null values for each column along with basic statistics.
    This function helps determine what preprocessing might be needed before applying machine learning.
    """
    logger.info(f"\nChecking {dataset_name} completeness:")
    total_rows = len(df)
    logger.info(f"Total rows: {total_rows}")
    
    # Calculate missing values per column
    missing_counts = df.isnull().sum()
    missing_percentage = (missing_counts / total_rows) * 100
    
    # Create a summary DataFrame with missing counts, percentage, and data types.
    summary = pd.DataFrame({
        "Missing Count": missing_counts,
        "Missing Percentage": missing_percentage,
        "Data Type": df.dtypes
    })
    
    logger.info(f"Missing Data Summary for {dataset_name}:\n{summary}")
    
    # Display basic descriptive statistics for further insight.
    logger.info(f"Basic Statistics for {dataset_name}:\n{df.describe(include='all')}")
    
    return summary

if __name__ == "__main__":
    logger.info("Starting integration and validation of joint data...")
    integrated_data = integrate_joint_data()
    logger.info("Integrated data loaded and validated.")
    
    logger.info("Running final query to compute comprehensive biomechanical metrics...")
    final_metrics = run_final_query()
    
    logger.info("Final query executed successfully. Displaying head of results:")
    print(final_metrics.head())
    
    # New: Check the summarized dataset for missing values and other quality metrics.
    summary_report = check_summary_dataset(final_metrics)
    logger.info("Summary dataset check complete.")
    
    # Optionally, save final_metrics and summary_report to CSV files.
    final_metrics.to_parquet("../../data/processed/ml_datasets/summarized/final_summarized_biomechanical_profile.parquet", index=False)
    summary_report.to_parquet("../../data/processed/ml_datasets/summarized/summary_dataset_report.parquet")


INFO:__main__:Starting integration and validation of joint data...
INFO:__main__:Loading sessions from the last month...
C:\Users\GeoffreyHadfield\AppData\Local\Temp\ipykernel_14836\1861705273.py:97: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sessions = pd.read_sql(f"""
INFO:__main__:Loading trials for these sessions...
C:\Users\GeoffreyHadfield\AppData\Local\Temp\ipykernel_14836\1861705273.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  trials = pd.read_sql(f"""
INFO:__main__:Loading joint data tables...
C:\Users\GeoffreyHadfield\AppData\Local\Temp\ipykernel_14836\1861705273.py:126: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or databa